In [ ]:
#https://www.kaggle.com/datasets/wyattowalsh/basketball/data
#https://github.com/mpope9/nba-sql/blob/master/image/NBA-ER.jpg

In [1]:
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
import warnings
import re
import psycopg2

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

In [7]:
options = webdriver.FirefoxOptions()
options.add_argument('-headless')
driver = webdriver.Firefox(options = options)

The geckodriver version (0.33.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (129.0.0.2295); currently, geckodriver 0.35.0 is recommended for firefox 129.*, so it is advised to delete the driver in PATH and retry


In [ ]:
try:
    # Establece la conexión con la base de datos
    connection = psycopg2.connect(
        dbname="mydatabase",
        user="myuser",
        password="mysecretpassword",
        host="localhost",  # normalmente es 'localhost' si es local
        port="5432"  # normalmente es 5432
    )
    
    # Crea un cursor para realizar operaciones en la base de datos
    cursor = connection.cursor()
except Exception as error:
    print(f"Error al conectar con la base de datos: {error}")

In [9]:
def getTableIDS(url):
    driver.get(url)
    tables_id = driver.find_elements(By.XPATH, "//table[@id]")
    list_id_tables = []
    for table in tables_id:
        table_id = table.get_attribute("id")
        list_id_tables.append(table_id)
    return list_id_tables

In [ ]:
def delete_unnamed_columns(df):
    df = df.loc[:, ~df.columns.str.contains('Unnamed')]
    return df

In [ ]:
def get_keys_dictionary(diccionario):
    keys = set(diccionario.keys())
    for values in diccionario.values():
        if isinstance(values, dict):
            keys.update(get_keys_dictionary(values))
    return keys

In [ ]:
def check_missing_values(dictionary):
    for key, value in dictionary.items():
        if isinstance(value, dict):
            print(f"Recorriendo diccionario bajo la clave '{key}':")
            check_missing_values(value)  
        elif isinstance(value, pd.DataFrame): 
            print(f"Revisando DataFrame bajo la clave '{key}':")
            
            if value.isnull().values.any():
                print("¡Hay valores nulos en el DataFrame!")
                print(value)
            
            unnamed_columns = [col for col in value.columns if 'Unnamed' in col]
            if unnamed_columns:
                print(f"¡El DataFrame tiene columnas 'Unnamed': {unnamed_columns}")

            empty_columns = [col for col in value.columns if value[col].empty]
            if empty_columns:
                print(f"¡El DataFrame tiene columnas vacías: {empty_columns}")

In [ ]:
# NBA Standings que es como quedó la season con todos los equipos
years = list(range(2022, 2023))
dictionary_of_teams = {}
# Itera a través de cada identificador de tabla y guarda en un DataFrame
dataframes = []
for year in years:
    dictionary_of_teams[year] = {}
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_standings.html'
    table_ids = getTableIDS(url)
    time.sleep(3)
    for table_id in table_ids:
        dictionary_of_teams[year][table_id] = {}
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        if table_id == 'expanded_standings':
            new_header = df.iloc[0]
            df = df[1:]
            df.columns = new_header
            df.reset_index(drop=True, inplace=True)
            dictionary_of_teams[year][table_id] = df
        else:    
            dictionary_of_teams[year][table_id] = df

In [ ]:
#De aqui para arriba tenemos las estadisticas generales de los equipos
#De aqui para abajo sacaremos las estadisticas de los jugadores por equipo

In [20]:
years = list(range(2022, 2023))
dictionary_of_players = {}
dictionary_of_players_playoffs = {}
teams_NBA_list = ['ATL']
# teams_NBA_list =  ['ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND','LAC','LAL','MEM','MIA','MIL','MIN','NOP', 'NYK','OKC', 'ORL','PHI', 
#                    'PHO', 'POR','SAC','SAS','TOR','UTA','WAS']

for team in teams_NBA_list:
    dictionary_of_players[team] = {}
    dictionary_of_players_playoffs[team] = {}
    for year in years:
        dictionary_of_players[team][year] = {}
        dictionary_of_players_playoffs[team][year] = {}
        url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'
        table_ids = getTableIDS(url)
        time.sleep(3)
        for table_id in table_ids:
            if 'playoffs' in table_id:
                dictionary_of_players_playoffs[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if  table_id == 'playoffs_pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players_playoffs[team][year][table_id] = df
            else:
                dictionary_of_players[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if table_id == 'adj_shooting' or table_id == 'shooting' or table_id == 'pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players[team][year][table_id] = df
        

,Unnamed: 0,G,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,Team,82.0,19705,3401,7241,0.470,1056,2821,0.374,2345,4420,0.531,1485,1829,0.812,823,2783,3606,2017,587,348,972,1534,9343
1,Team/G,NaN,240.3,41.5,88.3,0.470,12.9,34.4,0.374,28.6,53.9,0.531,18.1,22.3,0.812,10.0,33.9,44.0,24.6,7.2,4.2,11.9,18.7,113.9
2,Lg Rank,NaN,30,11,13,7.000,11,18,2.000,14,13,18.000,3,11,4.000,17,17,20,15,22,23,1,7,6
3,Year/Year,NaN,-0.6%,1.7%,1.2%,0.002,3.6%,3.1%,0.002,0.8%,0.1%,0.004,-8.0%,-8.0%,-0.000,-4.9%,-3.2%,-3.6%,2.0%,2.5%,-10.7%,-10.4%,-3.2%,0.2%
4,Opponent,82.0,19705,3435,7294,0.471,1051,2888,0.364,2384,4406,0.541,1294,1634,0.792,838,2758,3596,2085,576,367,1046,1668,9215
5,Opponent/G,NaN,240.3,41.9,89.0,0.471,12.8,35.2,0.364,29.1,53.7,0.541,15.8,19.9,0.792,10.2,33.6,43.9,25.4,7.0,4.5,12.8,20.3,112.4
6,Lg Rank,NaN,30,24,19,23.000,21,17,25.000,21,17,21.000,6,5,27.000,13,11,13,23,5,10,27,6,21
7,Year/Year,NaN,-0.6%,1.5%,-0.2%,0.008,7.6%,3.1%,0.015,-0.9%,-2.3%,0.007,-6.8%,-5.5%,-0.011,2.3%,3.0%,2.8%,4.4%,-9.2%,-11.2%,3.7%,3.0%,0.9%


In [37]:
for team, year in dictionary_of_players.items():
    for year, tables in year.items():
        print(tables.keys())


# DE AQUI VER QUE TABLA INTERESA, E INTENTAR VER COMO ESCTRUCTURAR LA BASE DE DATOS
# MAYBE ESQUEMAS DE PLAYER Y TABLAS POR SEPARADO POR CADA COSA QUE SE QUIERA
# PERO ANTES DE NADA CREAR TABLAS CON LOS DATOS QUE QUERAMOS

dict_keys(['roster', 'team_and_opponent', 'team_misc', 'per_game', 'totals', 'per_minute', 'per_poss', 'advanced', 'adj_shooting', 'shooting', 'pbp', 'salaries2'])


In [26]:
dictionary_of_players_playoffs['ATL'][2022]['playoffs_per_game']

,Rk,Player,Age,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Trae Young,23,5,5,37.2,4.4,13.8,0.319,1.4,7.6,0.184,3.0,6.2,0.484,0.370,5.2,6.6,0.788,1.0,4.0,5.0,6.0,0.6,0.0,6.2,2.6,15.4
1,2,De'Andre Hunter,24,5,5,35.0,7.8,14.0,0.557,2.4,5.2,0.462,5.4,8.8,0.614,0.643,3.2,4.0,0.800,0.4,3.4,3.8,0.6,0.8,0.2,2.2,4.0,21.2
2,3,Kevin Huerter,23,5,5,30.8,3.4,9.4,0.362,1.8,6.2,0.290,1.6,3.2,0.500,0.457,0.6,0.8,0.750,0.2,2.8,3.0,3.8,1.2,0.6,2.2,3.0,9.2
3,4,Delon Wright,29,5,0,27.4,3.0,5.8,0.517,1.0,2.6,0.385,2.0,3.2,0.625,0.603,1.2,1.8,0.667,2.2,2.6,4.8,2.8,0.8,0.2,1.0,0.4,8.2
4,5,Bogdan Bogdanović,29,4,0,26.8,5.0,12.3,0.408,2.3,6.5,0.346,2.8,5.8,0.478,0.500,2.0,2.5,0.800,1.0,3.8,4.8,3.0,0.3,0.3,1.0,3.0,14.3
5,6,John Collins,24,5,4,24.4,3.8,7.8,0.487,0.8,2.2,0.364,3.0,5.6,0.536,0.538,1.0,2.0,0.500,0.4,4.2,4.6,1.2,0.4,0.2,1.0,2.4,9.4
6,7,Danilo Gallinari,33,5,3,22.4,3.6,9.0,0.400,0.8,3.0,0.267,2.8,6.0,0.467,0.444,2.2,2.2,1.000,0.4,3.8,4.2,0.8,0.2,0.0,1.0,1.2,10.2
7,8,Onyeka Okongwu,21,5,1,21.6,1.8,3.2,0.563,0.0,0.0,NaN,1.8,3.2,0.563,0.563,1.6,2.0,0.800,2.2,3.2,5.4,0.4,0.8,0.8,0.8,3.8,5.2
8,9,Clint Capela,27,2,2,20.0,1.0,3.0,0.333,0.0,0.0,NaN,1.0,3.0,0.333,0.333,0.0,0.0,NaN,2.0,5.5,7.5,0.0,0.5,0.5,0.5,1.0,2.0
9,10,Timothé Luwawu-Cabarrot,26,4,0,5.5,0.5,1.0,0.500,0.3,0.8,0.333,0.3,0.3,1.000,0.625,0.0,0.0,NaN,0.0,0.8,0.8,0.5,0.0,0.0,0.3,1.5,1.3


In [ ]:
check_missing_values(dictionary_of_players)

In [ ]:
#celda de limpieza de datos
dictionary_of_players['ATL'][2022]['roster'] = dictionary_of_players['ATL'][2022]['roster'].drop(columns=['Unnamed: 6'])
dictionary_of_players['ATL'][2022]['team_and_opponent'] = dictionary_of_players['ATL'][2022]['team_and_opponent'].rename(columns={'Unnamed: 0': ''})
dictionary_of_players['ATL'][2022]['team_misc'].columns = dictionary_of_players['ATL'][2022]['team_misc'].iloc[0]
dictionary_of_players['ATL'][2022]['team_misc'] = dictionary_of_players['ATL'][2022]['team_misc'][1:]
dictionary_of_players['ATL'][2022]['per_poss'] = dictionary_of_players['ATL'][2022]['per_poss'] .drop(columns=['Unnamed: 27'])
dictionary_of_players['ATL'][2022]['advanced'] = dictionary_of_players['ATL'][2022]['advanced'].drop(columns=['Unnamed: 17', 'Unnamed: 22'])

In [ ]:
#pabajo players vs equipos

In [ ]:
# url = 'https://www.basketball-reference.com/players/a/'
# driver.get(url)
# time.sleep(3)
# table_id = getTableIDS(url)[0]
# table_element = driver.find_element(By.ID, table_id)
# table_html = table_element.get_attribute('outerHTML')
# soup = BeautifulSoup(table_html, 'html.parser')
# filas = soup.find_all('tr')

# active_players = []
# for fila in filas:
#     if fila.find('strong'):
#         regex = re.compile(r'(?<=href=").*?(?=")')
#         href = regex.findall(str(fila))
#         href = href[0]
#         regex = re.compile(r'(?<=/).*(?=.html)')
#         player = regex.findall(href)
#         player = player[0]
#         resultado = re.search(r'[^/]+/([^/]+)$', player)
#         active_players.append(resultado.group(1))

# # Obtener la información de cada jugador activo
# dictionary_of_players_individually = {}

# for player in active_players:
#     url = f'https://www.basketball-reference.com/players/a/{player}.html'
#     driver.get(url)
#     time.sleep(1)  # Ajusta este tiempo según lo necesario, puede que no necesites tanto como 3 segundos
    
#     player_html = driver.page_source
#     player_soup = BeautifulSoup(player_html, 'html.parser')
    
#     player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
#     # Aquí puedes obtener otros datos del jugador según tu necesidad y agregarlos al diccionario
#     dictionary_of_players_individually[player_name] = {}  # Agregar los datos del jugador

# # Ahora tienes un diccionario con la información de cada jugador activo
# print(dictionary_of_players_individually)


In [12]:
url = 'https://www.basketball-reference.com/players/a/'
driver.get(url)
time.sleep(3)
table_id = getTableIDS(url)[0]
table_element = driver.find_element(By.ID, table_id)
table_html = table_element.get_attribute('outerHTML')
soup = BeautifulSoup(table_html, 'html.parser')
filas = soup.find_all('tr')
dictionary_of_players_individually = {}

active_players = []
for fila in filas:
    if fila.find('strong'):
        regex = re.compile(r'(?<=href=").*?(?=")')
        href = regex.findall(str(fila))
        href = href[0]
        regex = re.compile(r'(?<=/).*(?=.html)')
        player = regex.findall(href)
        player = player[0]
        resultado = re.search(r'[^/]+/([^/]+)$', player)
        active_players.append(resultado.group(1))
        
for player in active_players:
    url = f'https://www.basketball-reference.com/players/a/{player}.html'
    driver.get(url)
    time.sleep(3)
    player_html = driver.page_source
    player_soup = BeautifulSoup(player_html, 'html.parser')
    player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
    table_ids = getTableIDS(url)

In [ ]:
'''Para mi yo del futuro
la celda de abajo es el diccionario que querias hacer de 
letra a: todos los jugadores con ese apellido y sus estadisticas
y luego la otra celda es de limpieza
lo de arriba entiendo yo que si se acaba haciendo lo de abajo se acabara borrando'''

In [10]:
# URL base de la página
base_url = 'https://www.basketball-reference.com/players/'

# Letra que quieres buscar
letra = 'a'

# URL de la página de jugadores con la letra específica
url = f'{base_url}{letra}/'

driver.get(url)
time.sleep(3)

table_id = getTableIDS(url)[0]
table_element = driver.find_element(By.ID, table_id)
table_html = table_element.get_attribute('outerHTML')
soup = BeautifulSoup(table_html, 'html.parser')
filas = soup.find_all('tr')

active_players = []

for fila in filas:
    if fila.find('strong'):
        regex = re.compile(r'(?<=href=").*?(?=")')
        href = regex.findall(str(fila))
        href = href[0]
        regex = re.compile(r'(?<=/).*(?=.html)')
        player = regex.findall(href)
        player = player[0]
        resultado = re.search(r'[^/]+/([^/]+)$', player)
        active_players.append(resultado.group(1))

# Diccionario para almacenar las estadísticas de los jugadores con apellido que empieza por 'A'
stats_by_letter = {}

# Obtener las estadísticas de cada jugador activo
for player in active_players:
    player_url = f'{base_url}{letra}/{player}.html'
    driver.get(player_url)
    time.sleep(3)
    player_html = driver.page_source
    player_soup = BeautifulSoup(player_html, 'html.parser')
    player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
    table_ids = getTableIDS(player_url)
    player_stats = {}
    for table_id in table_ids:
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        player_stats[table_id] = df

    stats_by_letter.setdefault(letra, {})[player_name] = player_stats

In [18]:
stats_by_letter['a']['Precious Achiuwa']['per_game']

,Season,Age,Tm,Lg,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2020-21,21.0,MIA,NBA,PF,61.0,4.0,12.1,2.0,3.7,0.544,0.0,0.0,0.000,2.0,3.7,0.546,0.544,0.9,1.8,0.509,1.2,2.2,3.4,0.5,0.3,0.5,0.7,1.5,5.0,NaN
1,2021-22,22.0,TOR,NBA,C,73.0,28.0,23.6,3.6,8.3,0.439,0.8,2.1,0.359,2.9,6.1,0.468,0.486,1.1,1.8,0.595,2.0,4.5,6.5,1.1,0.5,0.6,1.2,2.1,9.1,NaN
2,2022-23,23.0,TOR,NBA,C,55.0,12.0,20.7,3.6,7.3,0.485,0.5,2.0,0.269,3.0,5.4,0.564,0.521,1.6,2.3,0.702,1.8,4.1,6.0,0.9,0.6,0.5,1.1,1.9,9.2,NaN
3,2023-24,24.0,TOT,NBA,"C,PF",74.0,18.0,21.9,3.2,6.3,0.501,0.4,1.3,0.268,2.8,5.0,0.562,0.529,0.9,1.5,0.616,2.6,4.0,6.6,1.3,0.6,0.9,1.1,1.9,7.6,NaN
4,2023-24,24.0,TOR,NBA,C,25.0,0.0,17.5,3.1,6.8,0.459,0.5,1.9,0.277,2.6,4.9,0.528,0.497,1.0,1.7,0.571,2.0,3.4,5.4,1.8,0.6,0.5,1.2,1.6,7.7,NaN
5,2023-24,24.0,NYK,NBA,PF,49.0,18.0,24.2,3.2,6.1,0.525,0.3,1.0,0.260,2.9,5.1,0.578,0.547,0.9,1.4,0.643,2.9,4.3,7.2,1.1,0.6,1.1,1.1,2.1,7.6,NaN
6,Career,NaN,NaN,NBA,NaN,263.0,62.0,19.9,3.1,6.5,0.481,0.4,1.4,0.307,2.7,5.1,0.528,0.514,1.1,1.8,0.608,1.9,3.7,5.7,1.0,0.5,0.6,1.0,1.9,7.8,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3 seasons,NaN,TOR,NBA,NaN,153.0,40.0,21.6,3.5,7.7,0.458,0.6,2.0,0.315,2.9,5.7,0.509,0.500,1.2,1.9,0.636,1.9,4.2,6.1,1.2,0.5,0.5,1.1,1.9,8.9,NaN
9,1 season,NaN,MIA,NBA,NaN,61.0,4.0,12.1,2.0,3.7,0.544,0.0,0.0,0.000,2.0,3.7,0.546,0.544,0.9,1.8,0.509,1.2,2.2,3.4,0.5,0.3,0.5,0.7,1.5,5.0,NaN


In [19]:
# Supongamos que tienes el diccionario stats_by_letter con las estadísticas de los jugadores

# Iterar sobre las claves principales del diccionario (en este caso, la letra 'A')
for letra, players_stats in stats_by_letter.items():
    print(f"Letra: {letra}")

    # Iterar sobre los jugadores y sus estadísticas
    for player, stats in players_stats.items():
        print(f"Jugador: {player}")
        # Imprimir las estadísticas de cada jugador
        for stat_name, value in stats.items():
            print(f"{stat_name}")

        print("\n")


Letra: a
Jugador: Precious Achiuwa
projection
per_game
playoffs_per_game
stathead_insights
totals
playoffs_totals
per_minute
playoffs_per_minute
per_poss
playoffs_per_poss
advanced
playoffs_advanced
adj_shooting
pbp
playoffs_pbp
shooting
playoffs_shooting
highs-reg-season
highs-playoffs
playoffs-series
sims-thru
sims-career
all_college_stats
all_salaries
stathead_table


Jugador: Steven Adams
per_game
playoffs_per_game
stathead_insights
totals
playoffs_totals
per_minute
playoffs_per_minute
per_poss
playoffs_per_poss
advanced
playoffs_advanced
adj_shooting
pbp
playoffs_pbp
shooting
playoffs_shooting
highs-reg-season
highs-playoffs
playoffs-series
sims-thru
sims-career
all_college_stats
all_salaries
contracts_hou


Jugador: Bam Adebayo
projection
per_game
playoffs_per_game
stathead_insights
totals
playoffs_totals
per_minute
playoffs_per_minute
per_poss
playoffs_per_poss
advanced
playoffs_advanced
adj_shooting
pbp
playoffs_pbp
shooting
playoffs_shooting
highs-reg-season
highs-playoffs
pla

In [ ]:
#todo
# limpiar el script
# creo que no hace falta selenium? probar a sacar todos los datos sin selenium
# sacar datos totales de cada jugador por season (check)
# sacar datos de cada jugador por equipo (check?)
# sacar datos de cada equipo por season
# asi de momento, a lo mejor hacer algo con sqlite mas adelante (no sqlite)
# separar home results away results
